# TATIANA — corpus preparation (run this on Colab, not on your laptop)

**What this does:** builds a public-mathematics corpus for Construction 5's Tier 0, embeds it,
and hands back two small `.npz` files. Nothing from `DOCS/` is uploaded — the corpus is fetched
from arXiv inside Colab, so no unpublished work leaves your machine.

**Why arXiv rather than the project docs.** Tier 0 asks whether the latent structure behind
co-activation is a **cover** (overlapping causes) or a **partition** (competing ones). arXiv
**cross-listing is real-world overlap**: a paper filed under both `math.AT` and `math.DG` is
literally one concept claimed by two topics. So this corpus plausibly *has* cover structure,
which makes a negative result informative instead of merely underpowered. The project docs
could only ever have told us about themselves.

**The one thing that must not drift:** the embedding model. It is pinned to the same
`BAAI/bge-small-en-v1.5` (384-d) that `embeddings.py` uses. A different model here produces
vectors that are silently meaningless against anything the engine computes later.

Runtime: a couple of minutes on a free CPU instance. No GPU needed.

In [ ]:
# ---------------------------------------------------------------------------
# OPTIONAL: pull the Python tooling straight from the repo instead of uploading
# files by hand. Solves the "Colab can't find cover.py" problem at the root --
# run_tier0.py imports cover.py and assembly_log.py, and hand-uploading is
# exactly the kind of thing that silently misses one.
#
# SPARSE CHECKOUT, MOS/python ONLY. Deliberate: a full clone would drag DOCS/
# onto Google's servers -- BLUEPRINT.pdf, FIRST DRAFT.pdf, the logbook -- which
# is precisely what fetching the corpus from arXiv instead of DOCS was meant to
# avoid. The Python half is all that runs here, so the rest has no reason to
# travel.
#
# NOTE: this fetches what is PUSHED. Anything sitting uncommitted on the laptop
# will not appear, and the failure looks identical to a missing file.
# ---------------------------------------------------------------------------

REPO   = "https://github.com/BelSonOfOm/TATIANA.git"
BRANCH = "main"          # or the working branch, e.g. fix-11-13-and-curvature-decisions

import os, subprocess, textwrap

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print(r.stdout, r.stderr)
    return r.returncode == 0

if not os.path.isdir("TATIANA"):
    # If the repo is PRIVATE this will fail on auth. Do NOT paste a token into
    # this cell -- put it in Colab's Secrets panel (key icon, left sidebar) and
    # read it with google.colab.userdata, so it never lands in notebook output
    # or in the saved .ipynb.
    ok = (sh(f"git clone --filter=blob:none --no-checkout --branch {BRANCH} {REPO} TATIANA")
          and sh("git sparse-checkout init --cone", cwd="TATIANA")
          and sh("git sparse-checkout set MOS/python", cwd="TATIANA")
          and sh("git checkout", cwd="TATIANA"))
    if not ok:
        print(textwrap.dedent("""
            Clone failed. Either the repo is private (use Colab Secrets for a PAT,
            never an inline token) or the branch name above is wrong.
            Falling back to hand-uploaded files is fine -- you need exactly:
                run_tier0.py, cover.py, assembly_log.py
        """).strip())

if os.path.isdir("TATIANA/MOS/python"):
    import sys
    sys.path.insert(0, "/content/TATIANA/MOS/python")
    print("tooling on sys.path:")
    for f in sorted(os.listdir("TATIANA/MOS/python")):
        if f.endswith(".py"):
            print("   ", f)
    # The import that actually matters -- fail here, loudly, rather than in cell 6.
    try:
        import cover, assembly_log
        print("\nimports OK: cover, assembly_log")
    except Exception as e:
        print(f"\nIMPORT FAILED: {e}")

In [ ]:
!pip -q install fastembed

# Must match python/embeddings.py exactly. Changing one REQUIRES changing the other.
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
EMBED_DIM   = 384

# Categories chosen to OVERLAP heavily. Picking six disjoint fields would plant a
# partition in the corpus and the test would be answering a question we rigged.
CATEGORIES  = ["math.AT", "math.DG", "math-ph", "quant-ph", "math.PR", "stat.ML"]
PER_CAT     = 200          # ~1200 concepts total
N_TASKS     = 3000

In [ ]:
import time, re, urllib.request, xml.etree.ElementTree as ET

ATOM = "{http://www.w3.org/2005/Atom}"

def fetch(cat, n):
    """arXiv's public API. No key, no auth. 3s between calls is their stated etiquette."""
    url = ("http://export.arxiv.org/api/query?"
           f"search_query=cat:{cat}&start=0&max_results={n}"
           "&sortBy=submittedDate&sortOrder=descending")
    with urllib.request.urlopen(url, timeout=60) as r:
        root = ET.fromstring(r.read())
    out = []
    for e in root.findall(f"{ATOM}entry"):
        title = " ".join((e.findtext(f"{ATOM}title") or "").split())
        summ  = " ".join((e.findtext(f"{ATOM}summary") or "").split())
        # Every category the paper is filed under -- this is the cross-listing,
        # recorded so overlap can be inspected later. It is NOT ground truth for
        # the fitted organs; the model never sees it.
        cats = [t.get("term") for t in e.findall(f"{ATOM}category")]
        if len(summ.split()) >= 40:
            out.append({"title": title, "abstract": summ, "cats": cats, "primary": cat})
    return out

papers, seen = [], set()
for cat in CATEGORIES:
    got = fetch(cat, PER_CAT)
    new = [p for p in got if p["title"] not in seen]
    for p in new:
        seen.add(p["title"])
    papers += new
    print(f"{cat:10s} fetched {len(got):4d}, new {len(new):4d}, total {len(papers)}")
    time.sleep(3)

n_cross = sum(1 for p in papers if len(set(p["cats"])) > 1)
print(f"\n{len(papers)} papers, {n_cross} cross-listed "
      f"({100*n_cross/max(len(papers),1):.0f}%) -- that fraction is the corpus's real overlap")

In [ ]:
# CONCEPTS = abstracts. QUERIES = sentences from them.
#
# The grain asymmetry is deliberate and load-bearing: a query at the same grain as
# the stored unit mostly retrieves its own source and nothing else, and a tick that
# retrieves ONE concept contributes no co-activation pair at all. A sentence is a
# plausible query; the abstract containing it plus its topical neighbours is a
# plausible retrieval.

def concept_name(p, i):
    """Short, readable, unique. This string is the vertex identity in the concept
    store and the column label in the assembly matrix, so an opaque hash would make
    every downstream table unreadable."""
    t = re.sub(r"[^\w\s\-/^+]", "", p["title"])[:70].strip()
    return f"{p['primary']}#{i:04d}: {t}"

concepts = [(concept_name(p, i), p["abstract"], p["primary"], ";".join(sorted(set(p["cats"]))))
            for i, p in enumerate(papers)]

tasks, seen_t = [], set()
for p in papers:
    for s in re.split(r"(?<=[.!?])\s+", p["abstract"]):
        s = s.strip()
        w = s.split()
        if not (6 <= len(w) <= 40):
            continue
        if sum(c.isalpha() for c in s) < len(s) * 0.6:
            continue
        k = s.lower()
        if k in seen_t:
            continue
        seen_t.add(k)
        tasks.append(s)

tasks = tasks[:N_TASKS]
print(f"concepts: {len(concepts)}")
print(f"tasks   : {len(tasks)}" + ("  <-- FEWER THAN REQUESTED; raise PER_CAT" if len(tasks) < N_TASKS else ""))

In [ ]:
import numpy as np
from fastembed import TextEmbedding

model = TextEmbedding(model_name=EMBED_MODEL)

def embed(texts, label):
    t0 = time.time()
    V = np.array(list(model.embed(texts)), dtype=np.float32)
    print(f"{label}: {V.shape} in {time.time()-t0:.1f}s")
    assert V.shape[1] == EMBED_DIM, f"got {V.shape[1]}-d, expected {EMBED_DIM}"
    # bge returns unit vectors. The engine relies on this: SearchOp sets
    # derived_variance = ||q||, so ||q|| != 1 silently shifts every retrieval.
    n = np.linalg.norm(V, axis=1)
    print(f"  norms in [{n.min():.4f}, {n.max():.4f}]")
    return V

CV = embed([c[1] for c in concepts], "concepts")
QV = embed(tasks, "queries  ")

In [ ]:
# Retrieval breadth, measured HERE so you do not discover it after a local run.
#
# The engine keeps a concept when squared Bures-Wasserstein <= max(0.1, 1/dim).
# With D_query = D_concept = 1 the epistemic term vanishes and that is ||q-c||^2 <= 0.1,
# i.e. cos >= 0.95 -- a NEAR-DUPLICATE filter. Construction 5 needs ticks with two or
# more concepts, so if this table says otherwise, the threshold is the binding
# constraint and must be set from these numbers rather than left at its default.

S = QV[:1000] @ CV.T
d2 = np.maximum(2.0 - 2.0 * S, 0.0)          # unit vectors => ||q-c||^2 = 2-2cos

print(f"{'epsilon':>8} {'mean/tick':>10} {'median':>7} {'% with >=2':>11} {'% empty':>8}")
print("-" * 48)
rows = []
for eps in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    k = (d2 <= eps).sum(axis=1)
    rows.append((eps, k.mean(), (k >= 2).mean()))
    tag = "  <- engine default" if eps == 0.1 else ""
    print(f"{eps:>8.2f} {k.mean():>10.2f} {np.median(k):>7.0f} "
          f"{100*(k>=2).mean():>10.1f}% {100*(k==0).mean():>7.1f}%{tag}")

usable = [e for e, m, p2 in rows if 3.0 <= m <= 12.0 and p2 > 0.8]
print(f"\nThresholds giving 3-12 concepts/tick with >80% pairs: "
      f"{usable if usable else 'NONE -- widen CATEGORIES or raise PER_CAT'}")
RECOMMENDED_EPS = usable[0] if usable else None
print(f"RECOMMENDED_EPS = {RECOMMENDED_EPS}")

In [ ]:
import json, os
from google.colab import files

np.savez_compressed(
    "concepts.npz",
    names=np.array([c[0] for c in concepts], dtype=object),
    texts=np.array([c[1] for c in concepts], dtype=object),
    primary=np.array([c[2] for c in concepts], dtype=object),
    crosslist=np.array([c[3] for c in concepts], dtype=object),
    vectors=CV,
)
np.savez_compressed(
    "queries.npz",
    texts=np.array(tasks, dtype=object),
    vectors=QV,
)
# Provenance travels WITH the vectors. A corpus whose origin cannot be
# reconstructed is not evidence, whatever the fit says.
with open("corpus_manifest.json", "w") as fh:
    json.dump({
        "source": "arXiv public API",
        "categories": CATEGORIES,
        "per_category_requested": PER_CAT,
        "n_concepts": len(concepts),
        "n_tasks": len(tasks),
        "n_cross_listed": int(n_cross),
        "embed_model": EMBED_MODEL,
        "embed_dim": EMBED_DIM,
        "recommended_eps": RECOMMENDED_EPS,
        "concept_grain": "abstract",
        "query_grain": "sentence from an abstract",
        "caveat": ("Queries are drawn from the same corpus as the concepts, so "
                   "retrieval is easier than against an outside question. That "
                   "inflates HOW MUCH is retrieved; it does not decide the SHAPE "
                   "of what is retrieved, which is what Tier 0 asks about."),
    }, fh, indent=2)

for f in ("concepts.npz", "queries.npz", "corpus_manifest.json"):
    print(f"{f:22s} {os.path.getsize(f)/1e6:6.2f} MB")

for f in ("concepts.npz", "queries.npz", "corpus_manifest.json"):
    files.download(f)

print("\nPut all three in MOS/python/, then locally:")
print("  python ingest_corpus.py --vectors concepts.npz")
print("  python accumulate.py --tasks-npz queries.npz --ticks 3000")
print("  python run_tier0.py")